In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
df=pd.read_csv("drug200.csv")
df.head()

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,drugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,drugY


In [3]:
label_encoders={}

for col in df.columns:
    if df[col].dtype=="object":
        le=LabelEncoder()
        df[col]=le.fit_transform(df[col])
        label_encoders[col]=le

In [4]:
X=df.drop("Drug", axis=1).values
y=df["Drug"].values

In [5]:
scaler=StandardScaler()
X=scaler.fit_transform(X)

In [6]:
class KNN:

    def __init__(self, k=3):
        self.k=k

    def fit(self, X, y):
        self.X_train=X
        self.y_train=y

    def predict(self, X):

        predictions=[]

        for x in X:

            distances=np.sqrt(np.sum((self.X_train-x)**2, axis=1))

            k_indices=np.argsort(distances)[:self.k]

            k_labels=self.y_train[k_indices]

            unique, counts=np.unique(k_labels, return_counts=True)

            predictions.append(unique[np.argmax(counts)])

        return np.array(predictions)

In [7]:
kf=KFold(n_splits=5, shuffle=True, random_state=42)

In [8]:
def evaluate_knn(k):

    acc_list=[]
    prec_list=[]
    rec_list=[]
    f1_list=[]

    for train_index, test_index in kf.split(X):

        X_train, X_test=X[train_index], X[test_index]
        y_train, y_test=y[train_index], y[test_index]

        model=KNN(k=k)

        model.fit(X_train, y_train)

        preds=model.predict(X_test)

        acc_list.append(accuracy_score(y_test, preds))
        prec_list.append(precision_score(y_test, preds, average="macro"))
        rec_list.append(recall_score(y_test, preds, average="macro"))
        f1_list.append(f1_score(y_test, preds, average="macro"))

    return np.mean(acc_list), np.mean(prec_list), np.mean(rec_list), np.mean(f1_list)

In [15]:
acc1, prec1, rec1, f11=evaluate_knn(1)

print("K=1")
print("Accuracy:", acc1)
print("Precision:", prec1)
print("Recall:", rec1)
print("F1 Score:", f11)

K=1
Accuracy: 0.9000000000000001
Precision: 0.8674920634920635
Recall: 0.9309548738155549
F1 Score: 0.8809783193002827


In [16]:
acc3, prec3, rec3, f13=evaluate_knn(3)

print("K=3")
print("Accuracy:", acc3)
print("Precision:", prec3)
print("Recall:", rec3)
print("F1 Score:", f13)

K=3
Accuracy: 0.845
Precision: 0.7856967607555843
Recall: 0.8284177466840006
F1 Score: 0.7880940442575592


c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
acc5, prec5, rec5, f15=evaluate_knn(5)

print("K=5")
print("Accuracy:", acc5)
print("Precision:", prec5)
print("Recall:", rec5)
print("F1 Score:", f15)

K=5
Accuracy: 0.7949999999999999
Precision: 0.7205816144639674
Recall: 0.777414936628559
F1 Score: 0.7284422687798038


c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [18]:
results = pd.DataFrame({
    "K": [1,3,5],
    "Accuracy": [acc1, acc3, acc5],
    "Precision": [prec1, prec3, prec5],
    "Recall": [rec1, rec3, rec5],
    "F1 Score": [f11, f13, f15]
})

results

,K,Accuracy,Precision,Recall,F1 Score
0,1,0.900,0.867492,0.930955,0.880978
1,3,0.845,0.785697,0.828418,0.788094
2,5,0.795,0.720582,0.777415,0.728442
